# Plant Disease Detection - Research Notebook

This notebook is for normal experimentation before moving code into the reusable pipeline. Use it to inspect the dataset, test model choices, train quickly on a subset, and check predictions.

In [ ]:
from pathlib import Path
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import yaml
from PIL import Image

print(tf.__version__)

## Load Project Settings

In [ ]:
with open('../config/config.yaml', encoding='utf-8') as f:
    config = yaml.safe_load(f)

with open('../params.yaml', encoding='utf-8') as f:
    params = yaml.safe_load(f)

config, params

## Download Dataset from Kaggle

Run this cell once. KaggleHub caches downloads, so later runs should be faster.

In [ ]:
import kagglehub

dataset_slug = config['data_ingestion']['source_URL']
download_path = Path(kagglehub.dataset_download(dataset_slug))
download_path

## Find Image Class Folders

In [ ]:
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(folder):
    return sorted([
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    ])

class_dirs = []
for path in download_path.rglob('*'):
    if path.is_dir() and list_images(path):
        class_dirs.append(path)

class_dirs = sorted(class_dirs)
print(len(class_dirs))
[path.name for path in class_dirs[:10]]

## Create a Small Balanced Research Subset

In [ ]:
subset_dir = Path('../artifacts/research_subset')
train_dir = subset_dir / 'train'
valid_dir = subset_dir / 'validation'

max_images_per_class = params['data_ingestion']['max_images_per_class']
validation_split = params['data_ingestion']['validation_split']
random_seed = params['data_ingestion']['random_seed']

if subset_dir.exists():
    shutil.rmtree(subset_dir)

for class_dir in class_dirs:
    images = list_images(class_dir)
    random.Random(random_seed).shuffle(images)
    selected = images[:max_images_per_class]
    split_index = int(len(selected) * (1 - validation_split))

    for split_name, split_images in [('train', selected[:split_index]), ('validation', selected[split_index:])]:
        target_dir = subset_dir / split_name / class_dir.name
        target_dir.mkdir(parents=True, exist_ok=True)
        for image_path in split_images:
            shutil.copy2(image_path, target_dir / image_path.name)

subset_dir

## Visualize Sample Images

In [ ]:
sample_paths = random.sample(list(train_dir.rglob('*.*')), 9)

plt.figure(figsize=(10, 10))
for idx, image_path in enumerate(sample_paths, start=1):
    plt.subplot(3, 3, idx)
    plt.imshow(Image.open(image_path))
    plt.title(image_path.parent.name, fontsize=8)
    plt.axis('off')
plt.tight_layout()

## Build TensorFlow Datasets

In [ ]:
image_size = tuple(params['prepare_base_model']['image_size'][:2])
batch_size = params['training']['batch_size']

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode='categorical',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    label_mode='categorical',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
num_classes, class_names[:5]

## Prepare Data Augmentation and Preprocessing

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (preprocess_input(augmentation(x, training=True)), y))
valid_ds = valid_ds.map(lambda x, y: (preprocess_input(x), y))

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
valid_ds = valid_ds.prefetch(tf.data.AUTOTUNE)

## Build Transfer Learning Model

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=tuple(params['prepare_base_model']['image_size']),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(params['prepare_base_model']['dropout_rate']),
    tf.keras.layers.Dense(num_classes, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(params['prepare_base_model']['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=params['training']['epochs'],
)

## Plot Training Curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.legend()
plt.title('Accuracy')

plt.show()

## Evaluate and Save Research Model

In [ ]:
loss, accuracy = model.evaluate(valid_ds)
print({'loss': loss, 'accuracy': accuracy})

research_model_path = Path('../artifacts/research_model.h5')
model.save(research_model_path)
research_model_path

## Try One Prediction

In [ ]:
sample_image = next(valid_dir.rglob('*.*'))
image = Image.open(sample_image).convert('RGB').resize(image_size)

array = np.expand_dims(np.asarray(image, dtype=np.float32), axis=0)
array = preprocess_input(array)
prediction = model.predict(array)[0]

predicted_class = class_names[int(np.argmax(prediction))]
confidence = float(np.max(prediction))

plt.imshow(image)
plt.axis('off')
plt.title(f'{predicted_class} ({confidence:.2%})')
plt.show()